Import Libraries

In [ ]:
import pandas as pd
import sqlite3
import json
from bs4 import BeautifulSoup

**TASK 1**

Select and naming files

In [ ]:
DB_FILE = "library.db"
JSON_FILE = "book_catalog.json"
HTML_FILE = "reading_kickoff.html"

Connect with database

In [ ]:
conn = sqlite3.connect(DB_FILE)

Know existed tables

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';",conn)
tables

Show Members Data

Import Libraries

In [ ]:
import pandas as pd
import sqlite3
import json
from bs4 import BeautifulSoup

**TASK 1**

Select and naming files

In [ ]:
DB_FILE = "library.db"
JSON_FILE = "book_catalog.json"
HTML_FILE = "reading_kickoff.html"

Connect with database

In [ ]:
conn = sqlite3.connect(DB_FILE)

Know existed tables

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';",conn)
tables

Show Members Data

In [ ]:
members = pd.read_sql_query("SELECT * FROM members;",conn)
members.head()

Checkouts

In [ ]:
checkouts = pd.read_sql_query("SELECT * FROM checkouts;",conn)
checkouts.head()

Books

In [ ]:
books = pd.read_sql_query("SELECT * FROM books;",conn)

books.head()



---



*Question 1*

In [ ]:
query1 = """
SELECT m.member_id,m.first_name,m.last_name,COUNT(c.checkout_id) AS checkout_count
FROM members AS m
LEFT JOIN checkouts AS c
ON m.member_id = c.member_id
GROUP BY
    m.member_id,
    m.first_name,
    m.last_name
ORDER BY m.member_id;
"""

answer1 = pd.read_sql_query(query1, conn)

answer1

*Question 2*

In [ ]:
pattern = "%a%"

query2 = f"""
SELECT
book_id,
title,
author
FROM books
WHERE author LIKE '{pattern}';
"""

answer2 = pd.read_sql_query(query2, conn)

answer2

*Question 3*

In [ ]:
query3 = """
SELECT
b.book_id,
b.title,
COUNT(c.checkout_id) AS checkout_count
FROM books AS b
JOIN checkouts AS c
ON b.book_id = c.book_id
GROUP BY
b.book_id,
b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""

answer3 = pd.read_sql_query(query3, conn)

answer3

*Question 4*

In [ ]:
query4 = """
SELECT
m.member_id,
m.first_name,
m.last_name,
COUNT(c.checkout_id) AS book_count
FROM members AS m
JOIN checkouts AS c
ON m.member_id = c.member_id
GROUP BY
m.member_id,
m.first_name,
m.last_name
ORDER BY book_count DESC
LIMIT 10;
"""

answer4 = pd.read_sql_query(query4, conn)

answer4

*Question 5*

In [ ]:
chosen_neighborhood = "Maadi"

query5 = f"""
SELECT
c.checkout_id,
c.member_id,
c.book_id,
c.checkout_date,
c.return_date
FROM checkouts AS c
JOIN members AS m
ON c.member_id = m.member_id
WHERE m.neighborhood = '{chosen_neighborhood}'
ORDER BY c.checkout_date DESC
LIMIT 10 OFFSET 10;
"""

answer5 = pd.read_sql_query(query5, conn)

answer5



---



Count total books for each member

In [ ]:
member_book_counts = (checkouts.groupby("member_id").size().reset_index(name="total_books_borrowed"))

member_book_counts.head()

Add book counts to members

In [ ]:
members_with_counts = members.merge(member_book_counts,on="member_id",how="left")

members_with_counts["total_books_borrowed"] = (members_with_counts["total_books_borrowed"].fillna(0).astype(int))

members_with_counts.head()

Merge Members with Chckouts

In [ ]:
combined_stage1 = checkouts.merge(
    members_with_counts,
    on="member_id",
    how="left"
)

combined_stage1.head()

Make sure not to miss checkouts

In [ ]:
print("Original checkouts:", len(checkouts))
print("After Stage 1:", len(combined_stage1))

Succeed



---



Read book Catalog

In [ ]:
with open(JSON_FILE, "r", encoding="utf-8") as f:
    catalog = json.load(f)

catalog_df = pd.DataFrame(catalog)

catalog_df.head()

Show book columns

In [ ]:
catalog_df.columns

Merge book details

In [ ]:
combined_stage2 = combined_stage1.merge(books,on="book_id",how="left")

combined_stage2.head()

Add json data

In [ ]:
combined_stage2 = combined_stage2.merge(catalog_df,on="book_id",how="left")

combined_stage2.head()

Verifying the number of rows

In [ ]:
print("Before adding book details:", len(combined_stage1))
print("After adding book details:", len(combined_stage2))



---



*Reading Kickoff HTML*

Open html

In [ ]:
with open(HTML_FILE, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

Extracting tables

In [ ]:
tables_html = pd.read_html(HTML_FILE)

len(tables_html)

show reading kickoff table

In [ ]:
kickoff = tables_html[0]

kickoff.head()

Knowing columns

In [ ]:
kickoff.columns

adding reading kickoff

In [ ]:
print(kickoff.columns)

In [ ]:
kickoff = kickoff.rename(columns={"Member ID": "member_id","Book ID": "book_id","Checkout Date": "checkout_date"})

kickoff_combined = kickoff.merge(members_with_counts,on="member_id",how="left")

kickoff_combined.head()

In [ ]:
print(kickoff_combined.columns.tolist())

Add books

In [ ]:
kickoff_combined = kickoff_combined.merge(books,on="book_id",how="left")

Add catalog

In [ ]:
kickoff_combined = kickoff_combined.merge(catalog_df,on="book_id",how="left")

Combine columns

In [ ]:
print(combined_stage2.columns.tolist())
print(kickoff_combined.columns.tolist())

In [ ]:
# Rename the book title and author columns if needed
kickoff_combined = kickoff_combined.rename(
    columns={
        "title_x": "title",
        "author_x": "author"
    }
)

# Remove duplicate columns only if they exist
kickoff_combined = kickoff_combined.drop(
    columns=["title_y", "author_y"],
    errors="ignore"
)

kickoff_combined.head()

Add missing checkout coloumns and arrange coloumns

In [ ]:
# Add missing checkout columns
kickoff_combined["checkout_id"] = pd.NA
kickoff_combined["return_date"] = pd.NA

# Arrange columns in the exact same order
kickoff_combined = kickoff_combined[
    combined_stage2.columns
]

print(kickoff_combined.columns.tolist())

In [ ]:
print("Database:", list(combined_stage2.columns))
print("Kickoff:", list(kickoff_combined.columns))
print(list(combined_stage2.columns) == list(kickoff_combined.columns))

In [ ]:
task1_combined_data = pd.concat([combined_stage2, kickoff_combined],ignore_index=True)

task1_combined_data.head()

In [ ]:
print("Database checkouts:", len(combined_stage2))
print("Reading Kickoff checkouts:", len(kickoff_combined))
print("Total combined records:", len(task1_combined_data))

In [ ]:
print(len(combined_stage2) + len(kickoff_combined)== len(task1_combined_data))

Save file

In [ ]:
task1_combined_data.to_csv("31010280200518-Library-task1_combined_data.csv",index=False)

In [ ]:
import os

file_name ="31010280200518-Library-task1_combined_data.csv"

print(os.path.exists(file_name))
print(os.path.getsize(file_name), "bytes")

---


Save SQL answers file

In [ ]:
sql_report = f"""
SQL QUESTION 1
How much is each member borrowing?

SQL QUERY:
{query1}

RESULT:
{answer1.to_string(index=False)}

------------------

SQL QUESTION 2
Which books match a chosen author pattern?

Chosen pattern: {pattern}

SQL QUERY:
{query2}

RESULT:
{answer2.to_string(index=False)}

--------------------------

SQL QUESTION 3
What are the five most popular books?

SQL QUERY:
{query3}

RESULT:
{answer3.to_string(index=False)}

---------------------

SQL QUESTION 4
Who are the ten most active readers?

SQL QUERY:
{query4}

RESULT:
{answer4.to_string(index=False)}

-----------------

SQL QUESTION 5
Neighborhood activity further back in time

Chosen neighborhood: {chosen_neighborhood}

SQL QUERY:
{query5}

RESULT:
{answer5.to_string(index=False)}

---------------------

WEB PAGE VS API REFLECTION

The Reading Kickoff data came from an HTML web page, which is designed mainly for people to read in a browser. An API, on the other hand, provides data directly to programs in a structured and predictable format.

For this project, the difference mattered because the Reading Kickoff data had to be extracted from an HTML table before it could be combined with the database data. If the same information had been provided through an API, it would likely have been easier to access and process automatically.
"""

with open("31010280200518-Library-task1_sql_answers.txt","w",encoding="utf-8") as f:
    f.write(sql_report)